# MILPA - Prototipo multiagente con CrewAI

Proyecto Agentes, Fase 3. Flujo secuencial: perfilamiento, dictamen normativo, escenarios y síntesis educativa.

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from crewai.tools import tool
from crewai import Agent, Task, Crew, Process, LLM

In [2]:
# Configuración del modelo de lenguaje
load_dotenv()

llm = LLM(
    model="anthropic/claude-haiku-5-5",
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

# Para usar el modelo local en lugar de Claude:
# llm = LLM(model="ollama/llama3.1", base_url="http://localhost:11434")

In [3]:
# Datos del usuario (caso de prueba con datos ficticios)
formulario = {
    "id_sesion": "demo-001",
    "nombre": "Rosa",
    "edad": 38,
    "ocupacion": "Vende tamales en un mercado de Puebla",
    "ingresos_semanales": [1800, 2200, 3500, 2600, 1900, 3100, 2800, 2400],
    "gastos_fijos_mensuales": 6500,
    "dependientes": 2,
    "fondo_emergencia_meses": 1,
    "meta": "Complementar su retiro a los 65 años",
    "edad_retiro": 65,
    "tolerancia_riesgo": "baja",
    "presenta_declaracion_anual": False,
    "tiene_cuenta_afore": True
}

with open("formulario_usuario.json", "w", encoding="utf-8") as f:
    json.dump(formulario, f, ensure_ascii=False, indent=2)

print(pd.Series(formulario))

id_sesion                                                             demo-001
nombre                                                                    Rosa
edad                                                                        38
ocupacion                                Vende tamales en un mercado de Puebla
ingresos_semanales            [1800, 2200, 3500, 2600, 1900, 3100, 2800, 2400]
gastos_fijos_mensuales                                                    6500
dependientes                                                                 2
fondo_emergencia_meses                                                       1
meta                                      Complementar su retiro a los 65 años
edad_retiro                                                                 65
tolerancia_riesgo                                                         baja
presenta_declaracion_anual                                               False
tiene_cuenta_afore                                  

In [4]:
# Tasas e inflación que se usan en las proyecciones
tasas = pd.DataFrame([
    {"instrumento": "Afore voluntaria", "tasa_anual": 8.0},
    {"instrumento": "PPR", "tasa_anual": 7.5},
    {"instrumento": "CETES", "tasa_anual": 7.0},
    {"instrumento": "Fondo de inversión", "tasa_anual": 6.5}
])

inflacion_anual = 4.0

# Valor de la UMA 2026 publicado por el INEGI
uma_anual = 42794.64

print(tasas.to_string(index=False))

       instrumento  tasa_anual
  Afore voluntaria         8.0
               PPR         7.5
             CETES         7.0
Fondo de inversión         6.5


In [5]:
def cargar_formulario():
    with open("formulario_usuario.json", encoding="utf-8") as f:
        return json.load(f)

In [6]:
# Convierte los ingresos semanales a mensuales y calcula cuánto puede ahorrar
def calcular_ingresos():
    datos = cargar_formulario()
    ingresos = datos["ingresos_semanales"]
    semanas = 4.33

    promedio = sum(ingresos) / len(ingresos)
    minimo = min(ingresos)
    maximo = max(ingresos)

    diferencias = []
    for x in ingresos:
        diferencias.append((x - promedio) ** 2)
    desviacion = (sum(diferencias) / len(ingresos)) ** 0.5

    ahorro = minimo * semanas - datos["gastos_fijos_mensuales"]
    if ahorro < 0:
        ahorro = 0

    return {
        "promedio_mensual": round(promedio * semanas),
        "minimo_mensual": round(minimo * semanas),
        "maximo_mensual": round(maximo * semanas),
        "variacion": round(desviacion / promedio, 2),
        "ahorro_mensual": round(ahorro),
        "como_se_calcula_el_ahorro": "ingreso del mes mas bajo menos gastos fijos"
    }

In [7]:
# Compara el riesgo que dice tener el usuario con el que puede asumir y se queda con el menor
def calcular_riesgo():
    datos = cargar_formulario()
    ingresos = calcular_ingresos()
    horizonte = datos["edad_retiro"] - datos["edad"]

    if datos["tolerancia_riesgo"] == "baja":
        declarado = "conservador"
    elif datos["tolerancia_riesgo"] == "media":
        declarado = "moderado"
    else:
        declarado = "dinamico"

    if datos["fondo_emergencia_meses"] < 3 or datos["dependientes"] >= 2:
        objetivo = "conservador"
    elif horizonte >= 15:
        objetivo = "dinamico"
    else:
        objetivo = "moderado"

    if declarado == "conservador" or objetivo == "conservador":
        perfil = "conservador"
    elif declarado == "moderado" or objetivo == "moderado":
        perfil = "moderado"
    else:
        perfil = "dinamico"

    return {
        "declarado": declarado,
        "objetivo": objetivo,
        "perfil_riesgo": perfil,
        "horizonte_anios": horizonte
    }

In [8]:
# Revisa qué instrumentos puede usar el usuario según su perfil
def calcular_elegibilidad():
    datos = cargar_formulario()
    riesgo = calcular_riesgo()
    tope = 5 * uma_anual

    if datos["presenta_declaracion_anual"]:
        beneficio = "Deducible hasta " + str(round(tope)) + " pesos (LISR, art. 151, fr. V)"
    else:
        beneficio = "No aplica porque no presenta declaración anual"

    filas = []

    if datos["tiene_cuenta_afore"]:
        filas.append({"instrumento": "Afore voluntaria", "estatus": "elegible",
                      "motivo": "Ya tiene cuenta Afore", "beneficio": beneficio,
                      "advertencia": "Si retira antes del retiro pierde el beneficio fiscal"})
    else:
        filas.append({"instrumento": "Afore voluntaria", "estatus": "no elegible",
                      "motivo": "No tiene cuenta Afore", "beneficio": beneficio,
                      "advertencia": "Primero tendría que abrir una cuenta"})

    filas.append({"instrumento": "PPR", "estatus": "elegible con condiciones",
                  "motivo": "Su ventaja principal es fiscal", "beneficio": beneficio,
                  "advertencia": "Es a largo plazo y sacar el dinero antes tiene costo fiscal"})

    filas.append({"instrumento": "CETES", "estatus": "elegible",
                  "motivo": "Se puede invertir desde 100 pesos", "beneficio": "No deducible",
                  "advertencia": "La tasa cambia en cada subasta"})

    if riesgo["perfil_riesgo"] == "conservador":
        filas.append({"instrumento": "Fondo de inversión", "estatus": "elegible con condiciones",
                      "motivo": "Perfil conservador: solo fondos de deuda", "beneficio": "No deducible",
                      "advertencia": "El valor del fondo puede bajar"})
    else:
        filas.append({"instrumento": "Fondo de inversión", "estatus": "elegible",
                      "motivo": "Va con su perfil de riesgo", "beneficio": "No deducible",
                      "advertencia": "El valor del fondo puede bajar"})

    return pd.DataFrame(filas)

In [9]:
# Calcula cuánto junta el usuario si ahorra cada mes hasta su retiro
def proyectar(ahorro_mensual, anios, tasa_anual, meses_sin_ahorrar):
    tasa_mensual = tasa_anual / 100 / 12
    saldo = 0
    aportado = 0

    for mes in range(anios * 12):
        saldo = saldo * (1 + tasa_mensual)
        if (mes % 12) < 12 - meses_sin_ahorrar:
            saldo = saldo + ahorro_mensual
            aportado = aportado + ahorro_mensual

    saldo_hoy = saldo / ((1 + inflacion_anual / 100) ** anios)
    return round(aportado), round(saldo), round(saldo_hoy)

In [10]:
# Arma tres escenarios por cada instrumento que sí puede usar
def calcular_escenarios():
    ingresos = calcular_ingresos()
    riesgo = calcular_riesgo()
    elegibilidad = calcular_elegibilidad()

    ahorro = ingresos["ahorro_mensual"]
    anios = riesgo["horizonte_anios"]
    meses_sin_ahorrar = round(ingresos["variacion"] * 12)

    filas = []
    for i in range(len(tasas)):
        instrumento = tasas.loc[i, "instrumento"]
        tasa = tasas.loc[i, "tasa_anual"]

        fila = elegibilidad[elegibilidad["instrumento"] == instrumento]
        if fila["estatus"].values[0] == "no elegible":
            continue

        escenarios = [
            ("conservadora", tasa - 1, meses_sin_ahorrar),
            ("base", tasa, round(meses_sin_ahorrar / 2)),
            ("optimista", tasa + 1, 0)
        ]

        for nombre, tasa_escenario, meses in escenarios:
            aportado, saldo, saldo_hoy = proyectar(ahorro, anios, tasa_escenario, meses)
            filas.append({"instrumento": instrumento, "escenario": nombre,
                          "tasa": tasa_escenario, "meses_sin_ahorrar": meses,
                          "total_aportado": aportado, "saldo_final": saldo,
                          "saldo_en_pesos_de_hoy": saldo_hoy})

    return pd.DataFrame(filas)

In [11]:
# Prueba de las funciones antes de usarlas en los agentes
print(calcular_ingresos())
print(calcular_riesgo())
print(calcular_escenarios().to_string(index=False))

{'promedio_mensual': 10987, 'minimo_mensual': 7794, 'maximo_mensual': 15155, 'variacion': 0.22, 'ahorro_mensual': 1294, 'como_se_calcula_el_ahorro': 'ingreso del mes mas bajo menos gastos fijos'}
{'declarado': 'conservador', 'objetivo': 'conservador', 'perfil_riesgo': 'conservador', 'horizonte_anios': 27}
       instrumento    escenario  tasa  meses_sin_ahorrar  total_aportado  saldo_final  saldo_en_pesos_de_hoy
  Afore voluntaria conservadora   7.0                  3          314442       936928                 324942
  Afore voluntaria         base   8.0                  2          349380      1238894                 429669
  Afore voluntaria    optimista   9.0                  0          419256      1769563                 613714
               PPR conservadora   6.5                  3          314442       859018                 297922
               PPR         base   7.5                  2          349380      1133346                 393063
               PPR    optimista   8.5  

In [12]:
# Definición de herramientas
@tool
def leer_formulario():
    """Devuelve los datos que el usuario escribió en el formulario."""
    return pd.Series(cargar_formulario()).to_string()

@tool
def obtener_ingresos():
    """Devuelve los ingresos mensuales del usuario y cuánto puede ahorrar al mes."""
    return pd.Series(calcular_ingresos()).to_string()

@tool
def obtener_riesgo():
    """Devuelve el perfil de riesgo del usuario y los años que le faltan para el retiro."""
    return pd.Series(calcular_riesgo()).to_string()

@tool
def obtener_elegibilidad():
    """Devuelve qué instrumentos de ahorro puede usar el usuario, con su motivo y advertencias."""
    return calcular_elegibilidad().to_string(index=False)

@tool
def obtener_uma():
    """Devuelve el valor anual de la UMA 2026 publicado por el INEGI."""
    return "UMA anual 2026: " + str(uma_anual) + " pesos (INEGI)"

@tool
def obtener_tasas():
    """Devuelve las tasas de cada instrumento y la inflación usada en las proyecciones."""
    return tasas.to_string(index=False) + "\nInflación anual: " + str(inflacion_anual) + "%"

@tool
def obtener_escenarios():
    """Devuelve los escenarios de ahorro hasta el retiro, en pesos de hoy."""
    return calcular_escenarios().to_string(index=False)

In [13]:
# Agentes
perfilador = Agent(
    role = "Analista de Perfil Financiero del Trabajador Informal",
    goal = (
        "Construir el perfil financiero del usuario e indicar qué datos faltan o son estimados."
    ),
    backstory = (
        "Fuiste asesor de educación financiera en programas comunitarios de Puebla. "
        "Sabes que los trabajadores informales cobran por día o por semana, así que nunca "
        "supones un dato que el usuario no dio. No recomiendas productos ni opinas sobre inversiones."
    ),
    tools = [leer_formulario, obtener_ingresos, obtener_riesgo],
    verbose = True,
    allow_delegation = False,
    llm = llm
)

auditor_normativo = Agent(
    role = "Auditor Normativo de Ahorro para el Retiro y Fiscalidad",
    goal = (
        "Definir qué instrumentos de ahorro puede usar el usuario y qué advertencias "
        "se le deben comunicar."
    ),
    backstory = (
        "Eres especialista en normatividad financiera y conoces la LISR y la LSAR. "
        "No citas límites ni requisitos de memoria: usas solo lo que te dan tus herramientas. "
        "Si necesitas aclarar un dato, se lo preguntas al analista de perfil y no a otro agente."
    ),
    tools = [obtener_elegibilidad, obtener_uma],
    verbose = True,
    allow_delegation = True,
    llm = llm
)

analista_escenarios = Agent(
    role = "Analista Actuarial de Escenarios de Ahorro para el Retiro",
    goal = (
        "Comparar escenarios de ahorro para los instrumentos autorizados, con supuestos claros."
    ),
    backstory = (
        "Eres actuario y trabajas con planes de retiro de personas con ingresos variables. "
        "No haces cálculos de memoria: todas las cifras salen de tus herramientas. "
        "Si tienes una duda sobre una regla, se la preguntas al auditor normativo."
    ),
    tools = [obtener_tasas, obtener_escenarios],
    verbose = True,
    allow_delegation = True,
    llm = llm
)

sintetizador = Agent(
    role = "Sintetizador Educativo Financiero",
    goal = "Explicar los resultados al usuario en lenguaje sencillo y sin recomendar un producto.",
    backstory = (
        "Eres educador financiero para público general. Explicas conceptos complicados de forma "
        "simple y nunca presentas una opción como la mejor."
    ),
    verbose = True,
    allow_delegation = False,
    llm = llm
)

In [14]:
# Revisa que la respuesta final traiga las seis secciones
secciones = [
    "Tu situación en pocas palabras",
    "Opciones que puedes considerar",
    "Lo que debes saber antes de decidir",
    "Datos que conviene confirmar",
    "Siguientes pasos",
    "Aviso"
]

def revisar_secciones(resultado):
    faltantes = []
    for seccion in secciones:
        if seccion.lower() not in resultado.raw.lower():
            faltantes.append(seccion)

    if faltantes:
        return (False, "Faltan estas secciones: " + ", ".join(faltantes))
    return (True, resultado.raw)

In [15]:
# Tareas
tarea_perfil = Task(
    name = "Perfil",
    description = (
        "Construye el perfil financiero del usuario. "
        "Usa la herramienta 'leer_formulario' para ver sus datos, 'obtener_ingresos' para saber "
        "cuánto gana al mes y cuánto puede ahorrar, y 'obtener_riesgo' para su perfil de riesgo. "
        "Indica qué datos son declarados y cuáles son estimados. "
        "No recomiendes instrumentos ni hagas proyecciones."
    ),
    expected_output = (
        "Un JSON con la edad, la ocupación, el ingreso mensual, los gastos fijos, los dependientes, "
        "los años que faltan para el retiro, el ahorro mensual posible, el perfil de riesgo y la "
        "lista de datos estimados."
    ),
    agent = perfilador,
    verbose = True
)

tarea_dictamen = Task(
    name = "Dictamen",
    description = (
        "Define qué instrumentos de ahorro se le pueden presentar al usuario: aportaciones "
        "voluntarias a la Afore, PPR, CETES y fondos de inversión. "
        "Usa 'obtener_elegibilidad' para el estatus y las advertencias de cada uno, y 'obtener_uma' "
        "para los topes en pesos. "
        "No calcules rendimientos ni ordenes los instrumentos por conveniencia."
    ),
    expected_output = (
        "Un JSON con cada instrumento, su estatus, el motivo, el beneficio fiscal y las advertencias."
    ),
    agent = auditor_normativo,
    context = [tarea_perfil],
    verbose = True
)

tarea_escenarios = Task(
    name = "Escenarios",
    description = (
        "Compara escenarios de ahorro solo para los instrumentos que el dictamen marcó como "
        "elegibles. Usa 'obtener_tasas' para las tasas y la inflación, y 'obtener_escenarios' para "
        "las proyecciones. Copia las cifras tal como te las da la herramienta y anota los supuestos. "
        "Compara por rendimiento, liquidez y riesgo, sin recomendar ninguno."
    ),
    expected_output = (
        "Una tabla con el instrumento, el escenario, el total aportado, el saldo final y el saldo "
        "en pesos de hoy, más la lista de supuestos."
    ),
    agent = analista_escenarios,
    context = [tarea_perfil, tarea_dictamen],
    verbose = True
)

tarea_sintesis = Task(
    name = "Sintesis",
    description = (
        "Escribe la respuesta final para el usuario, hablándole de tú, con frases cortas y "
        "cantidades en pesos de hoy. Usa solo la información del perfil, el dictamen y los escenarios. "
        "Organiza el texto en seis secciones con estos títulos: 'Tu situación en pocas palabras', "
        "'Opciones que puedes considerar', 'Lo que debes saber antes de decidir', "
        "'Datos que conviene confirmar', 'Siguientes pasos' y 'Aviso'. "
        "En el aviso menciona que MILPA da orientación educativa y no sustituye a un asesor "
        "financiero certificado. No presentes ninguna opción como la mejor."
    ),
    expected_output = "Un reporte en Markdown de máximo 600 palabras con las seis secciones.",
    agent = sintetizador,
    context = [tarea_perfil, tarea_dictamen, tarea_escenarios],
    guardrail = revisar_secciones,
    guardrail_max_retries = 2,
    output_file = "respuesta_final.md",
    verbose = True
)

In [16]:
# Configuración del Crew
crew = Crew(
    name = "Equipo_MILPA",
    description = "Equipo de agentes para orientación financiera y ahorro para el retiro",
    agents = [perfilador, auditor_normativo, analista_escenarios, sintetizador],
    tasks = [tarea_perfil, tarea_dictamen, tarea_escenarios, tarea_sintesis],
    process = Process.sequential,
    verbose = True,
    output_log_file = "log_crew.txt"
)

resultado = await crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: Equipo_MILPA                                                                                             │
│  ID: f17bc83f-d81a-4d3b-93c7-2a27a77265fe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Perfil                                                                                                   │
│  ID: dbe71dfe-512e-4a07-9e48-1276241fad53                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Perfil Financiero del Trabajador Informal                                                   │
│                                                                                                                 │
│  Task: Construye el perfil financiero del usuario. Usa la herramienta 'leer_formulario' para ver sus datos,     │
│  'obtener_ingresos' para saber cuánto gana al mes y cuánto puede ahorrar, y 'obtener_riesgo' para su perfil de  │
│  riesgo. Indica qué datos son declarados y cuáles son estimados. No recomiendes instrumentos ni hagas           │
│  proyecciones.                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: leer_formulario                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: obtener_ingresos                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: obtener_riesgo                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (29 outputs left out)

╭───────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Sintesis                                                                                                 │
│  Agent: Sintetizador Educativo Financiero                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: Equipo_MILPA                                                                                             │
│  ID: f17bc83f-d81a-4d3b-93c7-2a27a77265fe                                                                       │
│  Final Output: ## Tu situación en pocas palabras                                                                │
│                                                                                                                 │
│  Rosa, tienes 38 años y vendes tamales en un mercado de Puebla. Tu meta es complementar tu retiro a los 65      │
│  años, así que faltan 27 años. Tu perfil de riesgo es conservador.                                              │
│                                                                                                                 │
│  Tus ingresos cambian bastante. En las últimas 8 semanas fueron de 1,800 a 3,500 pesos por semana. Con tus      │
│  gastos fijos de 6,500 pesos al mes, el ahorro posible que estimamos es de 1,294 pesos al mes. Es una cifra     │
│  estimada, no un monto que hayas decidido ahorrar. Tienes 1 mes de fondo de emergencia y 2 dependientes.        │
│                                                                                                                 │
│  ## Opciones que puedes considerar                                                                              │
│                                                                                                                 │
│  Estas son las cuatro opciones que salieron elegibles para tu perfil. **Ninguna es la mejor para ti.** Cada     │
│  una tiene ventajas y riesgos distintos.                                                                        │
│                                                                                                                 │
│  - **Aportaciones voluntarias a la Afore:** ya tienes cuenta Afore. Es de largo plazo. Hoy no aplica beneficio  │
│  fiscal, porque no presentas declaración anual.                                                                 │
│  - **PPR (Plan Personal de Retiro):** es de largo plazo. Sus condiciones de elegibilidad todavía deben          │
│  verificarse. Sacar el dinero antes tiene costo fiscal.                                                         │
│  - **CETES:** puedes invertir desde 100 pesos. La tasa cambia en cada subasta. No tienen deducción fiscal.      │
│  - **Fondos de inversión:** por tu perfil conservador, solo se consideran fondos de deuda. Su valor puede       │
│  bajar. No son deducibles.                                                                                      │
│                                                                                                                 │
│  ## Lo que debes saber antes de decidir                                                                         │
│                                                                                                                 │
│  Las cifras siguientes son proyecciones con supuestos. No son garantías. Suponen aportar 1,294 pesos al mes     │
│  durante 27 años, en pesos de hoy, sin contar comisiones ni impuestos.                                          │
│                                                                                                                 │
│  | Opción | Conservador | Base | Optimista |                                                                    │
│  |---|---|---|---|                                                                                              │
│  | Afore voluntaria | 324,942 | 429,669 | 613,714 |   

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
print(resultado)

## Tu situación en pocas palabras

Rosa, tienes 38 años y vendes tamales en un mercado de Puebla. Tu meta es complementar tu retiro a los 65 años, así que faltan 27 años. Tu perfil de riesgo es conservador.

Tus ingresos cambian bastante. En las últimas 8 semanas fueron de 1,800 a 3,500 pesos por semana. Con tus gastos fijos de 6,500 pesos al mes, el ahorro posible que estimamos es de 1,294 pesos al mes. Es una cifra estimada, no un monto que hayas decidido ahorrar. Tienes 1 mes de fondo de emergencia y 2 dependientes.

## Opciones que puedes considerar

Estas son las cuatro opciones que salieron elegibles para tu perfil. **Ninguna es la mejor para ti.** Cada una tiene ventajas y riesgos distintos.

- **Aportaciones voluntarias a la Afore:** ya tienes cuenta Afore. Es de largo plazo. Hoy no aplica beneficio fiscal, porque no presentas declaración anual.
- **PPR (Plan Personal de Retiro):** es de largo plazo. Sus condiciones de elegibilidad todavía deben verificarse. Sacar el dinero ant

In [18]:
# Resultado de cada tarea
for salida in resultado.tasks_output:
    print("=====", salida.name, "=====")
    print(salida.raw, "\n")

===== Perfil =====
```json
{
  "id_sesion": "demo-001",
  "nombre": "Rosa",
  "edad": {
    "valor": 38,
    "origen": "declarado"
  },
  "ocupacion": {
    "valor": "Vende tamales en un mercado de Puebla",
    "origen": "declarado"
  },
  "ingreso_mensual": {
    "promedio_mensual": 10987,
    "minimo_mensual": 7794,
    "maximo_mensual": 15155,
    "variacion": 0.22,
    "ingresos_semanales_declarados": [1800, 2200, 3500, 2600, 1900, 3100, 2800, 2400],
    "promedio_semanal_calculado": 2537.5,
    "origen": "estimado a partir de ingresos semanales declarados; la conversión semana a mes (aprox. 4.33 semanas) es un supuesto del sistema"
  },
  "gastos_fijos_mensuales": {
    "valor": 6500,
    "origen": "declarado"
  },
  "dependientes": {
    "valor": 2,
    "origen": "declarado"
  },
  "anios_para_retiro": {
    "valor": 27,
    "edad_retiro_declarada": 65,
    "origen": "calculado (65 - 38) a partir de datos declarados; coincide con el horizonte reportado por el sistema"
  },
  "aho